In [1]:
from google.colab import drive
drive.mount("/content/drive")

from getpass import getpass
token = getpass("GitHub token: ")

!rm -rf IIB-Project
!git clone https://{token}@github.com/aharris64/IIB-Project.git

%cd /content/IIB-Project/cnn

DATA_ROOT = "/content/drive/MyDrive/IIB_Project/data"
RUNS_ROOT = "/content/drive/MyDrive/IIB_Project/runs"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GitHub token: ··········
Cloning into 'IIB-Project'...
remote: Enumerating objects: 452, done.
remote: Counting objects: 100% (452/452), done.
remote: Compressing objects: 100% (337/337), done.
remote: Total 452 (delta 206), reused 352 (delta 110), pack-reused 0 (from 0)
Receiving objects: 100% (452/452), 12.79 MiB | 13.98 MiB/s, done.
Resolving deltas: 100% (206/206), done.
/content/IIB-Project/cnn


In [2]:
import torch
import numpy as np
import torch.nn as nn
from pathlib import Path
import json
import random
from datetime import datetime
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report
import csv

from train import train
from evaluate import evaluate
from models import build_model
from load_data import get_dataloaders
import config

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # # Deterministic can slow things down but stricly reproducable
    # torch.backends.cudnn.deterministic = False
    # torch.backends.cudnn.benchmark = True

def trainable_parameters(model):
    return [p for p in model.parameters() if p.requires_grad]

def save_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(obj, f, indent=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(config.SEED)

# Load Datachange p
train_loader, val_loader, test_loader = get_dataloaders(
        root_folder=DATA_ROOT,
        dataset=config.DATASET,
        batch_size=config.BATCH_SIZE,
    )
train_ds = train_loader.dataset
print("Dataset: ".config.DATASET)

# Get Model
model = build_model(config.MODEL_NAME, config.NUM_CLASSES, config.FREEZE)
model = model.to(device)
print("Model: ", config.MODEL_NAME)

# Criterion and Optimizer
counts = np.bincount([y for _, y in train_ds.samples])
weights = counts.sum() / (len(counts) * counts)
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
print("Train class counts:", counts.tolist())
print("Class weights:", class_weights.tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    trainable_parameters(model),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY)

# Save Experiment
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path(RUNS_ROOT) / f"{config.MODEL_NAME}_{run_id}"
out_dir.mkdir(parents=True, exist_ok=True)

cfg_snapshot = {k: getattr(config, k) for k in dir(config) if k.isupper()}
save_json(out_dir / "config.json", cfg_snapshot)

run_meta = {
    "run_id": run_id,
    "timestamp": datetime.now().isoformat(),
    "device": str(device),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
    "seed": int(config.SEED),
    "train_class_counts": counts.tolist(),
    "class_weights": class_weights.detach().cpu().tolist(),
}
save_json(out_dir / "run_meta.json", run_meta)

# Train
best_epoch, best_state, history = train(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    config.NUM_EPOCHS,
    config.PATIENCE
)

# Evaluate best model
model.load_state_dict(best_state)
model.to(device)

val_loss, y_true_v, y_pred_v, y_prob_v = evaluate(model, val_loader, device, criterion)
test_loss, y_true_t, y_pred_t, y_prob_t = evaluate(model, test_loader, device, criterion)

# Save History
save_json(out_dir / "history.json", history)

# Metrics
results = {
    "best_epoch": int(best_epoch),
    "val": {
        "loss": float(val_loss),
        "macro_f1": float(f1_score(y_true_v, y_pred_v, average="macro")),
        "balanced_acc": float(balanced_accuracy_score(y_true_v, y_pred_v)),
        "confusion_matrix": confusion_matrix(y_true_v, y_pred_v).tolist(),
        "classification_report": classification_report(y_true_v, y_pred_v, digits=4),
    },
    "test": {
        "loss": float(test_loss),
        "macro_f1": float(f1_score(y_true_t, y_pred_t, average="macro")),
        "balanced_acc": float(balanced_accuracy_score(y_true_t, y_pred_t)),
        "confusion_matrix": confusion_matrix(y_true_t, y_pred_t).tolist(),
        "classification_report": classification_report(y_true_t, y_pred_t, digits=4),
    },
}
save_json(out_dir / "metrics.json", results)

# Save predictions
np.savez_compressed(out_dir / "predictions_val.npz",
                    y_true=y_true_v, y_pred=y_pred_v, y_prob=y_prob_v)
np.savez_compressed(out_dir / "predictions_test.npz",
                    y_true=y_true_t, y_pred=y_pred_t, y_prob=y_prob_t)

# Save best weights
torch.save(best_state, out_dir / "best_model_state_dict.pt")

print(f"Saved run artifacts to: {out_dir}")

def save_misclassified_csv(out_path: Path, dataset, y_true, y_pred):
    # dataset.samples is list of (filepath, class_index)
    rows = []
    for i, ((fp, _), yt, yp) in enumerate(zip(dataset.samples, y_true, y_pred)):
        if int(yt) != int(yp):
            filename = Path(fp).name
            rows.append([filename, int(yt), int(yp)])

    with out_path.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["filename", "y_true", "y_pred"])
        w.writerows(rows)

# NOTE: assumes val_loader.dataset.samples aligns with loader order (true if shuffle=False for val/test)
save_misclassified_csv(out_dir / "misclassified_val.csv", val_loader.dataset, y_true_v, y_pred_v)
save_misclassified_csv(out_dir / "misclassified_test.csv", test_loader.dataset, y_true_t, y_pred_t)

Class mapping: {'normal': 0, 'papilledema': 1, 'pseudopapilledema': 2}
Train size: 2457 Val size: 523 Test size: 539
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 195MB/s]


Model:  mobilenet_v3
Train class counts: [1748, 456, 253]
Class weights: [0.46853548288345337, 1.7960525751113892, 3.237154245376587]
[Epoch 1] Train loss: 0.6123 | Val loss: 0.3454 | Val macro-F1: 0.8262
[Epoch 2] Train loss: 0.3739 | Val loss: 0.2526 | Val macro-F1: 0.8515
[Epoch 3] Train loss: 0.3231 | Val loss: 0.2329 | Val macro-F1: 0.8711
[Epoch 4] Train loss: 0.2733 | Val loss: 0.2306 | Val macro-F1: 0.8716
[Epoch 5] Train loss: 0.2304 | Val loss: 0.2335 | Val macro-F1: 0.8689
[Epoch 6] Train loss: 0.2209 | Val loss: 0.2309 | Val macro-F1: 0.8642
[Epoch 7] Train loss: 0.2063 | Val loss: 0.2246 | Val macro-F1: 0.8677
[Epoch 8] Train loss: 0.2041 | Val loss: 0.2283 | Val macro-F1: 0.8685
[Epoch 9] Train loss: 0.2158 | Val loss: 0.2224 | Val macro-F1: 0.8689
[Epoch 10] Train loss: 0.1735 | Val loss: 0.2262 | Val macro-F1: 0.8640
[Epoch 11] Train loss: 0.1743 | Val loss: 0.2197 | Val macro-F1: 0.8726
[Epoch 12] Train loss: 0.1650 | Val loss: 0.2133 | Val macro-F1: 0.8851
[Epoch 13] 